# FLIGHT PREDICTION
This is a complex dataset because of catogerical data

Target feature: Price
Input features: Airline, Date_of_Journey, Source, Destination, Route, Dep_Time, Arrival_Time, Duration, Total_Stops, Additional_Info

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel("flight_price.xlsx")
df.head()
df.tail()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
10678,Air Asia,9/04/2019,Kolkata,Banglore,CCU → BLR,19:55,22:25,2h 30m,non-stop,No info,4107
10679,Air India,27/04/2019,Kolkata,Banglore,CCU → BLR,20:45,23:20,2h 35m,non-stop,No info,4145
10680,Jet Airways,27/04/2019,Banglore,Delhi,BLR → DEL,08:20,11:20,3h,non-stop,No info,7229
10681,Vistara,01/03/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,12648
10682,Air India,9/05/2019,Delhi,Cochin,DEL → GOI → BOM → COK,10:55,19:15,8h 20m,2 stops,No info,11753


In [3]:
df.shape
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 1.8 MB


,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


Since most features are objects, they need to be converted into numerical form via feature engineering before any model can use them.

## Feature 1 - Date_of_Journey -> Date/Month/Year
- Date_of_Journey is stored as text in DD/MM/YYYY format. Splitting on / separates it into three components.
- These new columns are still object type after creation, so they need explicit type conversion.
- After Splitting we dont need original column so we can drop it.

In [8]:
df['date'] = df['Date_of_Journey'].str.split('/').str[0]
df['month'] = df['Date_of_Journey'].str.split('/').str[1]
df['year'] = df['Date_of_Journey'].str.split('/').str[2]

df['date'] = df['date'].astype(int)
df['month'] = df['month'].astype(int)
df['year'] = df['year'].astype(int)

df.drop(labels='Date_of_Journey',axis='columns',inplace=True)
print(df.columns)

Index(['Airline', 'Source', 'Destination', 'Route', 'Dep_Time', 'Arrival_Time',
       'Duration', 'Total_Stops', 'Additional_Info', 'Price', 'date', 'month',
       'year'],
      dtype='str')


## Feature 2 - Arrival Time -> Arrival_Hour, Arrival_Minute
- Arrival Time is stored as hour:minute which can be seperated
- First store any trailing dates then split time
- They are also in object type so convert to int.
- Drop Original Column


In [19]:
df['Arrival_Time'] = df['Arrival_Time'].str.split(' ').str[0]
df['Arrival_Hour'] = df['Arrival_Time'].str.split(':').str[0]
df['Arrival_Min'] = df['Arrival_Time'].str.split(':').str[1]
df['Arrival_Hour'] = df['Arrival_Hour'].astype(int)
df['Arrival_Min'] = df['Arrival_Min'].astype(int)
df.drop(labels='Arrival_Time', axis='columns', inplace=True)
print(df.columns)

Index(['Airline', 'Source', 'Destination', 'Route', 'Dep_Time', 'Duration',
       'Total_Stops', 'Additional_Info', 'Price', 'date', 'month', 'year',
       'Arrival_Hour', 'Arrival_Min'],
      dtype='str')


## Feature 3 - Departure Time -> Departure_Hour, Departure_Minute
Same as arrival time


In [22]:
df['Dep_Hour'] = df['Dep_Time'].str.split(':').str[0]
df['Dep_Min'] = df['Dep_Time'].str.split(':').str[1]
df['Dep_Hour'] = df['Dep_Hour'].astype(int)
df['Dep_Min'] = df['Dep_Min'].astype(int)
df.drop(labels='Dep_Time', axis='columns', inplace=True)
print(df.columns)

Index(['Airline', 'Source', 'Destination', 'Route', 'Duration', 'Total_Stops',
       'Additional_Info', 'Price', 'date', 'month', 'year', 'Arrival_Hour',
       'Arrival_Min', 'Dep_Hour', 'Dep_Min'],
      dtype='str')


## Feature 4 - Duration -> Duration_Hour, Duration_Minute
- We will split hours and minutes.
- We cant directly use h or m as splitting delimeter because some flights dont have hours or minutes
- instead we will use a regex where in an entry: digits before h are classified as hour and digits before m are classified as minute
- If a flight dont have hours or minutes, we will fill with 0

In [29]:
df['Duration_Hour'] = df['Duration'].str.extract(r'(\d+)h').fillna(0).astype(int)
df['Duration_Minute'] = df['Duration'].str.extract(r'(\d+)m').fillna(0).astype(int)
print(df['Duration_Hour'])
print(df['Duration_Minute'])
df.drop(labels='Duration',axis=1,inplace=True)

0         2
1         7
2        19
3         5
4         4
         ..
10678     2
10679     2
10680     3
10681     2
10682     8
Name: Duration_Hour, Length: 10683, dtype: int64
0        50
1        25
2         0
3        25
4        45
         ..
10678    30
10679    35
10680     0
10681    40
10682    20
Name: Duration_Minute, Length: 10683, dtype: int64


## Feature 5 - Route
It is redundant since we have source and destination

In [30]:
df.drop(labels='Route', axis=1, inplace=True)
print(df.columns)

Index(['Airline', 'Source', 'Destination', 'Total_Stops', 'Additional_Info',
       'Price', 'date', 'month', 'year', 'Arrival_Hour', 'Arrival_Min',
       'Dep_Hour', 'Dep_Min', 'Duration_Hour', 'Duration_Minute'],
      dtype='str')


## Feature 6 - Stops
We can find unique stop values then map them

In [ ]:
print(df['Total_Stops'].unique())
stop_map = {
    'non-stop': 0,
    '1 stop': 1,
    '2 stops': 2,
    '3 stops': 3,
    '4 stops': 4,
    np.nan: 1   #becaue 1 is mode - ordinal encoding since theres intrinsic ranking
}
df['Total_Stops'] = df['Total_Stops'].map(stop_map)
print(df['Total_Stops'])

<ArrowStringArray>
['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops']
Length: 6, dtype: str
0        0
1        2
2        2
3        1
4        1
        ..
10678    0
10679    0
10680    0
10681    0
10682    2
Name: Total_Stops, Length: 10683, dtype: int64


## Other Features
Now we have other features left which can't be split to form numerical columns and are also not have intrinsic ranking so we can do one hot encoding.

In [39]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
encoded_arr = encoder.fit_transform(df[['Airline','Source','Destination','Additional_Info']]).toarray()
encoded_df = pd.DataFrame(
    encoded_arr,
    columns=encoder.get_feature_names_out()
) 
df= pd.concat([df.reset_index(drop=True), encoded_df], axis=1)
df.drop(['Airline', 'Source', 'Destination'], axis=1, inplace=True)
print(df)

       Total_Stops Additional_Info  Price  date  month  year  Arrival_Hour  \
0                0         No info   3897    24      3  2019             1   
1                2         No info   7662     1      5  2019            13   
2                2         No info  13882     9      6  2019             4   
3                1         No info   6218    12      5  2019            23   
4                1         No info  13302     1      3  2019            21   
...            ...             ...    ...   ...    ...   ...           ...   
10678            0         No info   4107     9      4  2019            22   
10679            0         No info   4145    27      4  2019            23   
10680            0         No info   7229    27      4  2019            11   
10681            0         No info  12648     1      3  2019            14   
10682            2         No info  11753     9      5  2019            19   

       Arrival_Min  Dep_Hour  Dep_Min  ...  Additional_Info_1 L

In [40]:
df.to_csv('10_flightPriceCleaned.csv', index=False)